In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_allc'
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snm3C/pseudobulk_allc'
def process_bed(s):
    dmrs_mc = pd.read_csv(f'{indir}/bed/features/liftover/{s}.dmrs_CGN-Both.full.bed', sep='\t', header=None)
    dmrs_mc.index = [f'{x}_{y}' for x,y in zip(dmrs_mc[0], dmrs_mc[1])]
    return dmrs_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, tmp_order), total=len(tmp_order)))

In [ ]:
[x.shape[0] for x in results]

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'

dmrs = pd.read_csv(f'{indir}/dmrs/methylpy/rna_cluster/rna_cluster_rms_results_collapsed.tsv',
                   sep='\t', header=0)
dmrs.index = [f'{x}_{y}' for x,y in zip(dmrs['#chr'], dmrs['start'])]
dmrs.shape

In [ ]:
dmrs.head()

In [ ]:
mc_results = [x[4] / x[5] for x in results]
dmr_df = pd.concat(mc_results, axis=1)
dmr_df.columns = tmp_order
dmr_df.shape

In [ ]:
dmr_df.head()

### start with fibro

In [ ]:
dmrs['hypermethylated_samples'].fillna('', inplace=True)
dmrs['hypomethylated_samples'].fillna('', inplace=True)

### let's look per cluster now

In [ ]:
hyper_df = pd.DataFrame(index=tmp_order)
hypo_df = pd.DataFrame(index=tmp_order)

for s in tmp_order:
    hyper_mask = [s in x for x in dmrs['hypermethylated_samples']]
    hypo_mask = [s in x for x in dmrs['hypomethylated_samples']]
    hyper_idx = dmrs[hyper_mask].index
    hyper_dmrs = dmr_df.loc[hyper_idx]

    hypo_idx = dmrs[hypo_mask].index
    hypo_dmrs = dmr_df.loc[hypo_idx]
    hyper_df[s] = hyper_dmrs.mean(axis=0)
    hypo_df[s] = hypo_dmrs.mean(axis=0)
    
hyper_df.columns = [f'{x}_hyper_DMRs' for x in hyper_df.columns]
hypo_df.columns = [f'{x}_hypo_DMRs' for x in hypo_df.columns]

hyper_df = hyper_df.T
hypo_df = hypo_df.T

In [ ]:
hyper_df

In [ ]:
hypo_df

In [ ]:
fig, axes = plt.subplots(1, figsize=(12,6))
sns.heatmap(hyper_df, vmin=0.2, vmax=0.8, cmap='coolwarm')

In [ ]:
fig, axes = plt.subplots(1, figsize=(12,6))
sns.heatmap(hypo_df, vmin=0.2, vmax=0.8, cmap='coolwarm')

### don't average, just include all DMRs?

In [ ]:
# tmp_hypo_df

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures'

hyper_df.to_csv(f'{outdir}/csv/figure1/snm3c_hyper_dmrs.csv', sep='\t')
hypo_df.to_csv(f'{outdir}/csv/figure1/snm3c_hypo_dmrs.csv', sep='\t')

In [ ]:
!date